# Exploratory Data Analysis of the Ames Housing Dataset

**Dataset:** House Prices: Advanced Regression Techniques (Kaggle), `train.csv` (about 1,460 rows x 81 columns).
**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn, SciPy.
**Objective:** Use exploratory data analysis only (no machine learning) to understand which factors drive house sale prices.

> **Before you run this notebook:** download `train.csv` from the Kaggle competition page and place it in the same folder as this notebook.

## Guiding questions
1. How is `SalePrice` distributed? Is it skewed?
2. Which numerical features correlate most strongly with `SalePrice`?
3. Does overall quality (`OverallQual`) change price in a clear, ordered way?
4. Do some neighborhoods consistently cost more than others?
5. Are there outliers, and are they errors or real but unusual sales?
6. Which columns have missing values, and which of those are not truly missing?

## Notebook map
| Section | Step |
|---|---|
| 2. Data loading | Step 1 |
| 3. Data understanding | Steps 2, 3 |
| 4. Data cleaning | Step 9 (run early because it affects later steps) |
| 5. Univariate analysis | Step 4 |
| 6. Bivariate / multivariate analysis | Steps 5, 6, 7 |
| 7. Outlier analysis | Step 8 |
| 8. Dashboard | Step 10 |
| 9. Conclusions | Insights and limitations |

## 2. Data loading (Step 1)
**Plan:** import the libraries, set a plotting style, load the CSV, and look at its shape, types and missing values.
Note: pandas reads the text `NA` as a missing value. In this dataset `NA` often means "feature absent", which we deal with in Section 4.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 6)
pd.set_option("display.max_columns", 100)

MAIN = "#2a6f97"      # main colour used in every panel
ACCENT = "#e07a1f"    # highlight colour (outliers, median lines)

df = pd.read_csv("train.csv")
print("Shape:", df.shape)
display(df.head())
df.info()

missing = df.isnull().sum().sort_values(ascending=False)
missing = missing[missing > 0]
missing_tbl = pd.DataFrame({"missing": missing, "percent": (missing / len(df) * 100).round(2)})
display(missing_tbl)
print(f"{df.shape[0]} rows, {df.shape[1]} columns, {len(missing)} columns with missing values")

**What we learned:** _(write your own observations here after running the cell above)_

Write a short note: number of rows and columns, number of columns with missing values, and your first impressions.

## 3. Data understanding
### Step 2: Classify variables
**Plan:** split columns into numerical, categorical and ordinal groups. `ExterQual` and `KitchenQual` become ordered categoricals (Po < Fa < TA < Gd < Ex). `OverallQual` and `OverallCond` are stored as numbers but are really ordinal, and `MSSubClass` is a numeric code that is really a nominal label.

In [ ]:
numeric_cols = df.select_dtypes(include="number").columns.tolist()
categorical_cols = df.select_dtypes(exclude="number").columns.tolist()

ordinal_map = {
    "ExterQual":   ["Po", "Fa", "TA", "Gd", "Ex"],
    "KitchenQual": ["Po", "Fa", "TA", "Gd", "Ex"],
}
for col, order in ordinal_map.items():
    df[col] = pd.Categorical(df[col], categories=order, ordered=True)

ORDINAL_NUMERIC = ["OverallQual", "OverallCond"]
NOMINAL_NUMERIC = ["MSSubClass"]

def classify(col):
    if col in ordinal_map or col in ORDINAL_NUMERIC:
        return "ordinal"
    if col in NOMINAL_NUMERIC:
        return "categorical (nominal)"
    if col in numeric_cols:
        if pd.api.types.is_integer_dtype(df[col]) and df[col].nunique() <= 30:
            return "numerical (discrete)"
        return "numerical (continuous)"
    return "categorical (nominal)"

summary = pd.DataFrame({
    "column": df.columns,
    "type": [classify(c) for c in df.columns],
    "n_unique": [df[c].nunique() for c in df.columns],
    "examples": [df[c].dropna().unique()[:3].tolist() for c in df.columns],
})
display(summary)
print(summary["type"].value_counts())

**What we learned:** _(write your own observations here after running the cell above)_

`OverallQual` is stored as a number but is really ordinal. Explain in your own words why that matters for how we summarise and plot it.

### Step 3: Descriptive statistics
**Plan:** for `SalePrice`, `GrLivArea` and `LotArea`, compute centre (mean, median, mode) and spread (std, variance, range, IQR, skewness, kurtosis), and compare mean to median. Then look at the most common categories.

In [ ]:
display(df[numeric_cols].describe().T)

rows = []
for col in ["SalePrice", "GrLivArea", "LotArea"]:
    s = df[col]
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    rows.append({
        "column": col, "mean": s.mean(), "median": s.median(), "mode": s.mode().iloc[0],
        "std": s.std(), "variance": s.var(), "range": s.max() - s.min(),
        "iqr": q3 - q1, "skew": s.skew(), "kurtosis": s.kurt(),
    })
stats_tbl = pd.DataFrame(rows).set_index("column")
display(stats_tbl.round(2))

for col in stats_tbl.index:
    m, md_ = stats_tbl.loc[col, "mean"], stats_tbl.loc[col, "median"]
    if m > md_:
        print(f"{col}: mean ({m:,.0f}) > median ({md_:,.0f}) -> right skewed")
    elif m < md_:
        print(f"{col}: mean ({m:,.0f}) < median ({md_:,.0f}) -> left skewed")
    else:
        print(f"{col}: mean equals median -> roughly symmetric")

for col in ["Neighborhood", "HouseStyle"]:
    print(f"\n--- {col} ---")
    print(df[col].value_counts())
    print("Mode:", df[col].mode().iloc[0])

**What we learned:** _(write your own observations here after running the cell above)_

**Interpretation prompt:** Explain what the gap between the mean and the median of `SalePrice` tells you.

## 4. Data cleaning (Step 9, run early)
**Plan:** check duplicates, inconsistent category labels, and missing values. In this dataset `NA` in many categorical columns means "no such feature" (no pool, no alley, no garage...), so we fill with `"None"` rather than dropping. Numeric gaps get a documented strategy. Every decision is printed so it can be written up.

Decisions made here:
- The spec lists `PoolQC, Alley, Fence, FireplaceQu, GarageType`; the same logic applies to the other garage, basement, masonry and misc columns, so they are included too.
- `LotFrontage`: median grouped by `Neighborhood` (similar lots sit in the same area).
- `MasVnrArea`: 0 (no masonry veneer means zero area).
- `GarageYrBlt`: set to `YearBuilt` where there is no garage (an assumption, flagged here so it is not forgotten).
- `Electrical`: the single missing value is filled with the mode.

In [ ]:
# --- duplicates (ignore the Id column, which is unique by design)
dup_count = df.drop(columns="Id").duplicated().sum()
print("Duplicate rows:", dup_count)
if dup_count > 0:
    df = df.drop_duplicates(subset=[c for c in df.columns if c != "Id"]).reset_index(drop=True)
    print("New shape:", df.shape)

# --- inconsistent labels in text columns
text_cols = [c for c in categorical_cols if c not in ordinal_map]
problems = {}
for col in text_cols:
    s = df[col].dropna().astype(str)
    cleaned = s.str.strip().str.lower()
    if cleaned.nunique() < s.nunique():
        problems[col] = (s.nunique(), cleaned.nunique())
if problems:
    print("Columns with casing/spacing problems:", problems)
    for col in problems:
        df[col] = df[col].astype("string").str.strip().str.lower()
else:
    print("No casing, spacing or spelling inconsistencies found in text columns.")

# --- NA means "feature absent"
absent_cols = ["PoolQC", "Alley", "Fence", "FireplaceQu", "GarageType", "MiscFeature",
               "GarageFinish", "GarageQual", "GarageCond", "BsmtQual", "BsmtCond",
               "BsmtExposure", "BsmtFinType1", "BsmtFinType2", "MasVnrType"]
absent_cols = [c for c in absent_cols if c in df.columns]
for col in absent_cols:
    n = df[col].isnull().sum()
    if n:
        print(f"{col}: {n} NA -> 'None' (feature absent)")
    df[col] = df[col].fillna("None")

# --- numeric columns
df["LotFrontage"] = df["LotFrontage"].fillna(df.groupby("Neighborhood")["LotFrontage"].transform("median"))
df["LotFrontage"] = df["LotFrontage"].fillna(df["LotFrontage"].median())   # safety net
df["MasVnrArea"] = df["MasVnrArea"].fillna(0)
df["GarageYrBlt"] = df["GarageYrBlt"].fillna(df["YearBuilt"])
if "Electrical" in df.columns:
    df["Electrical"] = df["Electrical"].fillna(df["Electrical"].mode().iloc[0])

# --- verify
remaining = df.isnull().sum()
print("\nRemaining missing values:", int(remaining.sum()))
print(remaining[remaining > 0])

# --- impossible values
print("\nYearBuilt > YrSold:", int((df["YearBuilt"] > df["YrSold"]).sum()))
print("YearRemodAdd < YearBuilt:", int((df["YearRemodAdd"] < df["YearBuilt"]).sum()))
area_cols = [c for c in numeric_cols if ("SF" in c or "Area" in c)]
neg = {c: int((df[c] < 0).sum()) for c in area_cols if (df[c] < 0).any()}
print("Negative areas:", neg if neg else "none")

**What we learned:** _(write your own observations here after running the cell above)_

Write down each cleaning decision and the reason for it. **Answers question 6:** which columns were truly missing and which were not?

## 5. Univariate analysis (Step 4)
**Plan:** look at the shape of `SalePrice` (histogram with KDE, boxplot, log-transformed histogram), then at price by quality and by neighborhood, and the counts of quality and house style.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(df["SalePrice"], kde=True, color=MAIN, ax=axes[0])
axes[0].set(title="SalePrice distribution", xlabel="SalePrice ($)", ylabel="Count")

sns.boxplot(x=df["SalePrice"], color=MAIN, ax=axes[1])
axes[1].set(title="SalePrice boxplot", xlabel="SalePrice ($)")

log_price = np.log(df["SalePrice"])
sns.histplot(log_price, kde=True, color=MAIN, ax=axes[2])
axes[2].set(title="log(SalePrice) distribution", xlabel="log(SalePrice)", ylabel="Count")

plt.tight_layout()
plt.show()

print(f"Skewness before log: {df['SalePrice'].skew():.3f}")
print(f"Skewness after log:  {log_price.skew():.3f}")

**What we learned:** _(write your own observations here after running the cell above)_

**Answers question 1.** Is `SalePrice` skewed? Did the log transform make it close to normal? Quote both skewness values.

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x="OverallQual", y="SalePrice", color=MAIN)
plt.title("SalePrice by OverallQual")
plt.xlabel("OverallQual (1 = very poor, 10 = very excellent)")
plt.ylabel("SalePrice ($)")
plt.show()

order = df.groupby("Neighborhood")["SalePrice"].median().sort_values().index
plt.figure(figsize=(14, 6))
sns.boxplot(data=df, x="Neighborhood", y="SalePrice", order=order, color=MAIN)
plt.xticks(rotation=90)
plt.title("SalePrice by Neighborhood (sorted by median)")
plt.xlabel("Neighborhood")
plt.ylabel("SalePrice ($)")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(data=df, x="OverallQual", color=MAIN, ax=axes[0])
axes[0].set(title="Houses per OverallQual", xlabel="OverallQual", ylabel="Count")
hs_order = df["HouseStyle"].value_counts().index
sns.countplot(data=df, x="HouseStyle", order=hs_order, color=MAIN, ax=axes[1])
axes[1].set(title="Houses per HouseStyle", xlabel="HouseStyle", ylabel="Count")
plt.tight_layout()
plt.show()

print(df.groupby("OverallQual")["SalePrice"].agg(["count", "median", "std"]).round(0))
print()
print(df.groupby("Neighborhood")["SalePrice"].median().sort_values(ascending=False).head(5))
print(df.groupby("Neighborhood")["SalePrice"].median().sort_values().head(5))

**What we learned:** _(write your own observations here after running the cell above)_

**Answers questions 3 and 4.** Does price rise steadily with quality? Does the spread change at the top? Which neighborhoods have the highest and lowest medians?

## 6. Bivariate and multivariate analysis
### Step 5: Correlation and covariance
**Plan:** compute covariance and Pearson correlation on the numeric columns (excluding `Id`), rank features by absolute correlation with `SalePrice`, and compare covariance with correlation for the top three.

In [ ]:
num_for_corr = [c for c in df.select_dtypes(include="number").columns if c != "Id"]   # recomputed after cleaning
cov_matrix = df[num_for_corr].cov()
corr_matrix = df[num_for_corr].corr(method="pearson")

top_corr = corr_matrix["SalePrice"].drop("SalePrice").abs().sort_values(ascending=False)
top10 = top_corr.head(10).index.tolist()
print("Top 10 features by |correlation| with SalePrice:")
print(corr_matrix.loc[top10, "SalePrice"].round(3))

print("\nTop 3: covariance vs correlation with SalePrice")
for col in top10[:3]:
    print(f"{col:14s} cov = {cov_matrix.loc[col, 'SalePrice']:>16,.1f}   corr = {corr_matrix.loc[col, 'SalePrice']:.3f}")

# Optional: Spearman for ordinal columns (ordered categories converted to codes)
ord_df = pd.DataFrame({
    "OverallQual": df["OverallQual"],
    "ExterQual": df["ExterQual"].cat.codes,
    "KitchenQual": df["KitchenQual"].cat.codes,
    "SalePrice": df["SalePrice"],
})
print("\nSpearman correlation with SalePrice (ordinal columns):")
print(ord_df.corr(method="spearman")["SalePrice"].drop("SalePrice").round(3))

# Correlated pair of predictors
print(f"\nGarageArea vs GarageCars correlation: {df['GarageArea'].corr(df['GarageCars']):.3f}")

**What we learned:** _(write your own observations here after running the cell above)_

**Answers question 2.** Name the strongest correlates.

**Interpretation prompts:**
- Why is correlation easier to compare than covariance? (Hint: units.)
- Correlation is not causation. `GarageArea` and `GarageCars` are correlated with each other. Why does that matter when you read the top-10 list?

### Step 6: Scatter plots and pair plots
**Plan:** scatter `GrLivArea`, `TotalBsmtSF` and `YearBuilt` against `SalePrice`, highlighting very large houses with low prices, then a pair plot of five key columns coloured by `CentralAir`.

In [ ]:
flag = (df["GrLivArea"] > 4000) & (df["SalePrice"] < 300000)

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(df.loc[~flag, "GrLivArea"], df.loc[~flag, "SalePrice"], alpha=0.5, color=MAIN, label="Other sales")
ax.scatter(df.loc[flag, "GrLivArea"], df.loc[flag, "SalePrice"], color=ACCENT, s=80, edgecolor="black",
           label="GrLivArea > 4000 and price < $300k")
ax.set(title="GrLivArea vs SalePrice", xlabel="GrLivArea (sq ft)", ylabel="SalePrice ($)")
ax.legend()
plt.show()
print("Flagged points:", int(flag.sum()))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.scatterplot(data=df, x="TotalBsmtSF", y="SalePrice", alpha=0.5, color=MAIN, ax=axes[0])
axes[0].set(title="TotalBsmtSF vs SalePrice", xlabel="TotalBsmtSF (sq ft)", ylabel="SalePrice ($)")
sns.scatterplot(data=df, x="YearBuilt", y="SalePrice", alpha=0.5, color=MAIN, ax=axes[1])
axes[1].set(title="YearBuilt vs SalePrice", xlabel="YearBuilt", ylabel="SalePrice ($)")
plt.tight_layout()
plt.show()

selected = ["SalePrice", "GrLivArea", "TotalBsmtSF", "GarageArea", "YearBuilt"]
g = sns.pairplot(df[selected + ["CentralAir"]], hue="CentralAir", diag_kind="kde",
                 palette={"Y": MAIN, "N": ACCENT}, plot_kws={"alpha": 0.5, "s": 20})
g.figure.suptitle("Pair plot of key features", y=1.02)
plt.show()

**What we learned:** _(write your own observations here after running the cell above)_

Which relationships look linear? Where do the flagged points sit relative to the main trend? Do they distort a trend line?

### Step 7: Correlation heatmap
**Plan:** one clear picture of how the top-10 features relate to `SalePrice` and to each other (upper triangle hidden).

In [ ]:
cols = ["SalePrice"] + top10
corr_top = df[cols].corr()
mask = np.triu(np.ones_like(corr_top, dtype=bool))

plt.figure(figsize=(10, 8))
sns.heatmap(corr_top, annot=True, fmt=".2f", cmap="coolwarm", center=0, mask=mask,
            linewidths=0.5, cbar_kws={"label": "Pearson correlation"})
plt.title("Top 10 features correlated with SalePrice")
plt.tight_layout()
plt.show()

**What we learned:** _(write your own observations here after running the cell above)_

Which pairs of features are strongly correlated with each other (possible redundancy)?

## 7. Outlier analysis (Step 8)
**Plan:** detect outliers in `GrLivArea` and `SalePrice` with the IQR rule and the z-score rule, compare counts and overlap, and decide which points to remove. The result is `df_clean`; the original `df` is left untouched.

In [ ]:
def iqr_outliers(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return series[(series < lower) | (series > upper)]

def zscore_outliers(series, threshold=3):
    z = (series - series.mean()) / series.std()
    return series[z.abs() > threshold]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, col in zip(axes, ["GrLivArea", "SalePrice"]):
    a = iqr_outliers(df[col])
    b = zscore_outliers(df[col])
    overlap = len(set(a.index) & set(b.index))
    print(f"{col}: IQR flags {len(a)}, z-score flags {len(b)}, overlap {overlap}")
    sns.boxplot(x=df[col], color=MAIN, ax=ax, flierprops={"markerfacecolor": ACCENT, "markeredgecolor": ACCENT})
    ax.set_title(f"{col}: boxplot (outliers marked)")
plt.tight_layout()
plt.show()

iqr_idx = set(iqr_outliers(df["GrLivArea"]).index) | set(iqr_outliers(df["SalePrice"]).index)
z_idx = set(zscore_outliers(df["GrLivArea"]).index) | set(zscore_outliers(df["SalePrice"]).index)
flagged = df.index.isin(list(iqr_idx | z_idx))

plt.figure(figsize=(9, 6))
plt.scatter(df.loc[~flagged, "GrLivArea"], df.loc[~flagged, "SalePrice"], alpha=0.5, color=MAIN, label="Not flagged")
plt.scatter(df.loc[flagged, "GrLivArea"], df.loc[flagged, "SalePrice"], alpha=0.8, color=ACCENT, label="Flagged by IQR or z-score")
plt.title("GrLivArea vs SalePrice with flagged outliers")
plt.xlabel("GrLivArea (sq ft)")
plt.ylabel("SalePrice ($)")
plt.legend()
plt.show()

# Decision: remove very large houses sold very cheaply (likely non-typical sales, e.g. partial sales).
# Large AND expensive houses are real, so they are kept.
remove = (df["GrLivArea"] > 4000) & (df["SalePrice"] < 300000)
print("\nRows removed:", int(remove.sum()))
display(df.loc[remove, ["Id", "GrLivArea", "SalePrice", "SaleCondition", "Neighborhood", "OverallQual"]])
df_clean = df[~remove].copy()
print("df shape:", df.shape, "-> df_clean shape:", df_clean.shape)

**What we learned:** _(write your own observations here after running the cell above)_

**Answers question 5.** Are the removed points errors or real but unusual sales? Why were large, expensive houses kept? Check the `SaleCondition` of the removed rows.

**Interpretation prompt:** Why does the z-score method flag fewer points on a skewed variable than the IQR method? (Hint: the long tail inflates the mean and the standard deviation.)

## 8. Dashboard (Step 10)
**Plan:** four panels in one figure (price by year built, top neighborhoods, price distribution with mean and median, and living area vs price coloured by quality), saved as `eda_dashboard.png`. The dashboard uses `df_clean`.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel 1: line plot
avg_by_year = df_clean.groupby("YearBuilt")["SalePrice"].mean()
axes[0, 0].plot(avg_by_year.index, avg_by_year.values, color=MAIN, label="Average price")
axes[0, 0].set(title="Average SalePrice by year built", xlabel="YearBuilt", ylabel="Average SalePrice ($)")
axes[0, 0].legend()

# Panel 2: horizontal bar plot (top 10 neighborhoods)
top_nb = df_clean.groupby("Neighborhood")["SalePrice"].mean().sort_values(ascending=False).head(10).sort_values()
axes[0, 1].barh(top_nb.index, top_nb.values, color=MAIN)
axes[0, 1].set(title="Top 10 neighborhoods by average price", xlabel="Average SalePrice ($)", ylabel="Neighborhood")

# Panel 3: histogram + KDE with mean and median
sns.histplot(df_clean["SalePrice"], kde=True, color=MAIN, ax=axes[1, 0])
mean_p, median_p = df_clean["SalePrice"].mean(), df_clean["SalePrice"].median()
axes[1, 0].axvline(mean_p, color=ACCENT, linestyle="--", label=f"Mean: ${mean_p:,.0f}")
axes[1, 0].axvline(median_p, color="black", linestyle=":", label=f"Median: ${median_p:,.0f}")
axes[1, 0].set(title="SalePrice distribution", xlabel="SalePrice ($)", ylabel="Count")
axes[1, 0].legend()

# Panel 4: scatter coloured by OverallQual
sc = sns.scatterplot(data=df_clean, x="GrLivArea", y="SalePrice", hue="OverallQual",
                     palette="viridis", alpha=0.7, ax=axes[1, 1])
axes[1, 1].set(title="GrLivArea vs SalePrice by OverallQual", xlabel="GrLivArea (sq ft)", ylabel="SalePrice ($)")
axes[1, 1].legend(title="OverallQual", fontsize=8, ncol=2)

fig.suptitle("House Price EDA Summary", fontsize=18, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig("eda_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()

## Verifying the expected insights
The project brief lists insights to **verify, not assume**. This cell computes a check for each one from the data, so the conclusions below rest on numbers.

In [ ]:
price = df["SalePrice"]
log_skew = np.log(price).skew()
corr_price = df[[c for c in df.select_dtypes(include="number").columns if c != "Id"]].corr()["SalePrice"].drop("SalePrice").abs().sort_values(ascending=False)
med_by_q = df.groupby("OverallQual")["SalePrice"].median()
std_by_q = df.groupby("OverallQual")["SalePrice"].std()
nb_med = df.groupby("Neighborhood")["SalePrice"].median().sort_values(ascending=False)
mask_unusual = (df["GrLivArea"] > 4000) & (df["SalePrice"] < 300000)

def trend_slope(d):
    return np.polyfit(d["GrLivArea"], d["SalePrice"], 1)[0]

checks = [
    ("1. SalePrice right skewed (skew > 0.75)", price.skew() > 0.75, f"skew = {price.skew():.2f}"),
    ("1b. log makes it close to normal (|skew| < 0.5)", abs(log_skew) < 0.5, f"log skew = {log_skew:.2f}"),
    ("2. OverallQual and GrLivArea in top-5 correlates",
     {"OverallQual", "GrLivArea"} <= set(corr_price.head(5).index), f"top 5 = {corr_price.head(5).index.tolist()}"),
    ("3. Median price rises with every quality level", bool(med_by_q.is_monotonic_increasing),
     f"medians = {med_by_q.astype(int).tolist()}"),
    ("3b. Spread is larger at top quality than bottom", std_by_q.dropna().iloc[-3:].mean() > std_by_q.dropna().iloc[:3].mean(),
     f"std low-3 = {std_by_q.dropna().iloc[:3].mean():,.0f}, std high-3 = {std_by_q.dropna().iloc[-3:].mean():,.0f}"),
    ("4. Top 3 neighborhoods' median > 1.4x overall median", nb_med.head(3).min() > 1.4 * price.median(),
     f"top 3 = {nb_med.head(3).astype(int).to_dict()}, overall median = {price.median():,.0f}"),
    ("5. Large cheap houses exist and flatten the trend line", int(mask_unusual.sum()) > 0,
     f"{int(mask_unusual.sum())} points; slope with = {trend_slope(df):.1f}, without = {trend_slope(df[~mask_unusual]):.1f} $/sq ft"),
]
for name, ok, detail in checks:
    print(("CONFIRMED  " if ok else "NOT CONFIRMED  ") + name + "  ->  " + detail)

## 9. Conclusions
_Fill this in using your own results. The expected insights from the brief are listed so you can confirm or reject each one with evidence from the cell above._

| # | Expected insight | Confirmed? | Evidence (numbers / plot) |
|---|---|---|---|
| 1 | `SalePrice` is right skewed; a log transform makes it close to normal | | |
| 2 | `OverallQual` and `GrLivArea` are among the strongest correlates of price | | |
| 3 | Price rises steadily with quality, with larger spread at the top | | |
| 4 | A few neighborhoods have clearly higher medians than the rest | | |
| 5 | A handful of very large, low-price houses are unusual sales that distort trend lines | | |
| 6 | Many apparent missing values are "feature absent" and should not be dropped | | |

### Answers to the guiding questions
1. Distribution of `SalePrice`:
2. Strongest numerical correlates:
3. Effect of `OverallQual`:
4. Neighborhood differences:
5. Outliers (errors or real?):
6. Missing values (truly missing vs absent feature):

### Limitations
- Correlation does not imply causation; several features are correlated with each other (e.g. `GarageArea` and `GarageCars`).
- Pearson correlation only captures linear relationships.
- Fill strategies in Section 4 are assumptions (especially `GarageYrBlt`).
- The data covers one city (Ames, Iowa) and one period (2006-2010), so conclusions may not generalise.
- This is exploratory analysis only; no predictive model was built or validated.